# Xiaomi Price Checker

Notebook controller for reading the Excel configuration, running the enabled website scrapers, collecting variant-level price/availability data, and writing the results to `RawData`.

The actual website scraping remains in `scrapers/` and is executed through `run_price_check.py`.


In [16]:
# ============================================================
# 1. SETUP
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys

from openpyxl import load_workbook
import pandas as pd

project_folder = Path(
    r"C:\Users\New\.vscode\Python_projects\XM_Price_Checker"
)

excel_file = project_folder / "XM_Price_Checker_Python.xlsm"

print("Project folder:", project_folder)
print("Excel exists:", excel_file.exists())


Project folder: C:\Users\New\.vscode\Python_projects\XM_Price_Checker
Excel exists: True


In [17]:
# ============================================================
# 2. LOAD EXCEL WORKBOOK
# ============================================================

workbook = load_workbook(
    excel_file,
    keep_vba=True
)

products_sheet = workbook["Products"]
links_sheet = workbook["Links"]
rawdata_sheet = workbook["RawData"]
results_sheet = workbook["Results"]

print("Workbook loaded successfully.")
print("Sheets:", workbook.sheetnames)
print("Products rows:", products_sheet.max_row)
print("Links rows:", links_sheet.max_row)
print("RawData rows:", rawdata_sheet.max_row)
print("Results rows:", results_sheet.max_row)


Workbook loaded successfully.
Sheets: ['Products', 'Links', 'RawData', 'Results', 'Settings']
Products rows: 31
Links rows: 141
RawData rows: 1
Results rows: 31


In [18]:
# ============================================================
# 3. LOAD PRODUCTS
# ============================================================

products = {}

for row in products_sheet.iter_rows(
    min_row=2,
    values_only=True
):
    if not row:
        continue

    model = row[1]
    name = row[2]
    ram = row[3]
    storage = row[4]

    if model is None or ram is None or storage is None:
        continue

    model = str(model).strip()
    ram = str(ram).strip()
    storage = str(storage).strip()

    target_id = f"{model}-{ram}-{storage}"

    products[target_id.upper()] = {
        "target_id": target_id,
        "name": str(name).strip() if name is not None else model,
        "model": model,
        "ram": ram,
        "storage": storage
    }

print("Products loaded:", len(products))

if products:
    print("Example:", next(iter(products.values())))


Products loaded: 29
Example: {'target_id': 'SOMALIAA-4-64', 'name': 'Redmi A7 pro', 'model': 'SOMALIAA', 'ram': '4', 'storage': '64'}


In [19]:
# ============================================================
# 4. LOAD ENABLED LINKS
# ============================================================

links = []

for row in links_sheet.iter_rows(
    min_row=2,
    values_only=True
):
    if not row:
        continue

    target_id = row[0]
    website = row[1]
    url = row[2]
    enabled = row[3]

    if (
        target_id
        and website
        and url
        and str(enabled).strip().upper() == "YES"
    ):
        links.append({
            "target_id": str(target_id).strip(),
            "website": str(website).strip().upper(),
            "url": str(url).strip()
        })

target_ids = sorted({
    item["target_id"].strip().upper()
    for item in links
})

print("Enabled links:", len(links))
print("Target IDs:", target_ids)


Enabled links: 80
Target IDs: ['O19AE-6-128', 'O19AE-8-256', 'P15AE-4-128', 'P16-8-256', 'P16U-12-512', 'P16U-8-256', 'P17-6-128', 'P6-8-256', 'P7E-8-256', 'SOMALIAA-4-64']


In [20]:
# ============================================================
# 5. RUN PRICE CHECKER
# ============================================================

def run_price_check(website, url, product):
    """Run the website scraper in a separate Python process."""

    script = project_folder / "run_price_check.py"

    product_json = json.dumps(
        product,
        ensure_ascii=False
    )

    process = subprocess.run(
        [
            sys.executable,
            str(script),
            website,
            url,
            product_json
        ],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )

    # Keep scraper/browser logs visible in the notebook.
    print(process.stdout)

    if process.returncode != 0:
        print(process.stderr)
        raise RuntimeError(
            f"{website} price check failed."
        )

    for line in process.stdout.splitlines():
        if line.startswith("RESULT_JSON:"):
            return json.loads(
                line.replace(
                    "RESULT_JSON:",
                    "",
                    1
                ).strip()
            )

    raise ValueError(
        f"No RESULT_JSON returned for {website}."
    )


In [21]:
# ============================================================
# 6. PRODUCT LOOKUP
# ============================================================

def get_product(target_id):
    """Return product information loaded from the Products sheet."""

    key = str(target_id).strip().upper()

    try:
        return products[key]
    except KeyError:
        raise ValueError(
            f"TargetID not found in Products: {target_id}"
        )


In [22]:
# ============================================================
# 7. PRICE COLLECTION
# ============================================================

all_results = []

for target_id in target_ids:

    product = get_product(target_id)

    print()
    print("=" * 60)
    print("TARGET:", target_id)
    print(
        "Product:",
        product["name"],
        "| RAM:", product["ram"],
        "| Storage:", product["storage"]
    )
    print("=" * 60)

    target_links = [
        item for item in links
        if item["target_id"].strip().upper() == target_id
    ]

    for item in target_links:

        website = item["website"]
        url = item["url"]

        print()
        print("-" * 50)
        print("Website:", website)
        print("URL:", url)

        try:
            results = run_price_check(
                website,
                url,
                product
            )

            print("SCRAPER RESULTS:", results)

            # No matching product is not a scraper error.
            if not results:
                print("No matching products found.")
                continue

            for result in results:
                all_results.append({
                    "run_time": datetime.now(),
                    "target_id": target_id,
                    "website": website,
                    "product_name": result.get(
                        "product_name",
                        product["name"]
                    ),
                    "variant": result.get("variant", ""),
                    "ram": result.get("ram", product["ram"]),
                    "storage": result.get("storage", product["storage"]),
                    "url": url,
                    "price": result.get("price"),
                    "currency": "PLN",
                    "availability": result.get(
                        "availability",
                        "Unknown"
                    ),
                    "status": "OK",
                    "error": ""
                })

        except Exception as e:
            print("ERROR:", str(e))

            all_results.append({
                "run_time": datetime.now(),
                "target_id": target_id,
                "website": website,
                "product_name": product["name"],
                "variant": "",
                "ram": product["ram"],
                "storage": product["storage"],
                "url": url,
                "price": None,
                "currency": "PLN",
                "availability": "Unknown",
                "status": "ERROR",
                "error": str(e)
            })

print()
print("=" * 60)
print("PRICE COLLECTION COMPLETE")
print("=" * 60)
print("Total results:", len(all_results))



TARGET: O19AE-6-128
Product: Redmi 15 | RAM: 6 | Storage: 128

--------------------------------------------------
Website: MEX
URL: https://www.mediaexpert.pl/search?query[menu_item]=&query[querystring]=Redmi%2015%206%2F128
AVANS MODULE PATH: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\scrapers\avans.py
NEONET MODULE PATH: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\scrapers\neonet.py
Website: MEX
URL: https://www.mediaexpert.pl/search?query[menu_item]=&query[querystring]=Redmi%2015%206%2F128
Product: {'target_id': 'O19AE-6-128', 'name': 'Redmi 15', 'model': 'O19AE', 'ram': '6', 'storage': '128'}
Opening: https://www.mediaexpert.pl/search?query[menu_item]=&query[querystring]=Redmi%2015%206%2F128
Page loaded.
Loading all Media Expert products...
Current h2/h3 count: 43
Current h2/h3 count: 51
Current h2/h3 count: 63
Current h2/h3 count: 63
Current h2/h3 count: 63
Finished loading products. Total h2/h3: 63
Total h2/h3 elements: 63

PRODUCT: Smartfon XIAOMI Redmi 15 6

In [23]:
# ============================================================
# 8. RESULTS / VERIFICATION
# ============================================================

results_df = pd.DataFrame(all_results)

print("Raw scraper results:", len(results_df), "rows")

if not results_df.empty:
    display(
        results_df[
            [
                "target_id",
                "website",
                "product_name",
                "variant",
                "ram",
                "storage",
                "price",
                "availability",
                "status",
                "error"
            ]
        ]
    )

    print()
    print("=" * 70)
    print("TARGET / WEBSITE SUMMARY")
    print("=" * 70)

    summary_df = (
        results_df
        .groupby(
            ["target_id", "website"],
            dropna=False
        )
        .agg(
            variants=("variant", "count"),
            available=(
                "availability",
                lambda x: (x == "Available").sum()
            ),
            unavailable=(
                "availability",
                lambda x: (x == "Unavailable").sum()
            ),
            prices=(
                "price",
                lambda x: x.notna().sum()
            ),
            errors=(
                "status",
                lambda x: (x == "ERROR").sum()
            )
        )
        .reset_index()
    )

    display(summary_df)
else:
    print("No scraper results were collected.")


Raw scraper results: 305 rows


,target_id,website,product_name,variant,ram,storage,price,availability,status,error
0,O19AE-6-128,MEX,Redmi 15,Black,6,128,649.0,Available,OK,
1,O19AE-6-128,MEX,Redmi 15,Purple,6,128,NaN,Unavailable,OK,
2,O19AE-6-128,MEX,Redmi 15,Titanium,6,128,NaN,Unavailable,OK,
3,O19AE-6-128,ELECTRO.PL,Redmi 15,Black,6 GB,128 GB,NaN,Unavailable,OK,
4,O19AE-6-128,ELECTRO.PL,Redmi 15,Purple,6 GB,128 GB,NaN,Unavailable,OK,
...,...,...,...,...,...,...,...,...,...,...
300,SOMALIAA-4-64,NEONET,Redmi A7 pro,Black,4 GB,64 GB,449.0,Available,OK,
301,SOMALIAA-4-64,NEONET,Redmi A7 pro,Blue,4 GB,64 GB,NaN,Available,OK,
302,SOMALIAA-4-64,MAX ELECTRO,Redmi A7 pro,Blue,4 GB,64 GB,399.0,Available,OK,
303,SOMALIAA-4-64,MAX ELECTRO,Redmi A7 pro,Black,4 GB,64 GB,399.0,Available,OK,



TARGET / WEBSITE SUMMARY


,target_id,website,variants,available,unavailable,prices,errors
0,O19AE-6-128,AVANS,3,0,3,0,0
1,O19AE-6-128,ELECTRO.PL,3,0,3,0,0
2,O19AE-6-128,KTR,1,1,0,1,0
3,O19AE-6-128,MAX ELECTRO,1,0,1,0,0
4,O19AE-6-128,MEX,3,1,2,1,0
...,...,...,...,...,...,...,...
75,SOMALIAA-4-64,MAX ELECTRO,3,3,0,3,0
76,SOMALIAA-4-64,MEX,3,3,0,3,0
77,SOMALIAA-4-64,MSH,3,2,1,3,0
78,SOMALIAA-4-64,NEONET,3,3,0,2,0


In [24]:
# ============================================================
# 9. RAW DATA OUTPUT
# ============================================================
#
# The RawData sheet is append-only.
# If you want a fresh run instead of keeping historical rows,
# clear the existing RawData rows manually before running this cell.
# ============================================================

for _, result in results_df.iterrows():

    price = result["price"]

    # Pandas NaN -> blank Excel cell
    if pd.isna(price):
        price = None

    rawdata_sheet.append([
        result["run_time"],
        result["target_id"],
        result["website"],
        result["product_name"],
        result["variant"],
        result["url"],
        price,
        result["currency"],
        result["availability"],
        result["status"],
        result["error"]
    ])

print("RawData rows written:", len(results_df))


RawData rows written: 305


In [ ]:
# ============================================================
# 10. GENERATE RESULTS SHEET
# ============================================================
#
# The Results sheet is the final report.
#
# IMPORTANT:
# - Matching between Products and RawData uses TargetID.
# - RRP is displayed but is NOT used for the comparison.
# - Only "RRP after Promotion" is used as the benchmark.
# - Offer start/end dates are copied to the report but are NOT used.
#
# Report logic for each website:
#
# 1. If all returned variants are Unavailable -> "unavailable"
# 2. Otherwise ignore unavailable variants and compare the
#    available prices with RRP after Promotion.
# 3. If the lowest available price == promotion price -> "OK"
#    (even if another colour is more expensive).
# 4. If the lowest available price is below promotion price ->
#    "<price> <colour>".
# 5. If the lowest available price is above promotion price ->
#    "<price> <colour>" unless every available colour has the
#    same price, in which case only "<price>" is returned.
# ============================================================

# 10.1 NORMALIZE NUMBER

def normalize_number(value):
    """
    Return a numeric value as float, or None.
    """

    if value is None or pd.isna(value):
        return None

    try:
        return float(value)

    except (TypeError, ValueError):
        return None


# 10.2 BUILD ONE WEBSITE REPORT VALUE

def build_report_value(
    raw_rows,
    promotion_price
):
    """
    Build the value that will be written into one
    website cell in Results H:O.
    """

    # --------------------------------------------------------
    # No scraper record at all
    # --------------------------------------------------------

    if not raw_rows:
        return ""

    promotion_price = normalize_number(
        promotion_price
    )

    # --------------------------------------------------------
    # Separate available and unavailable variants
    # --------------------------------------------------------

    available_rows = []

    unavailable_count = 0

    for row in raw_rows:

        availability = str(
            row.get("availability", "")
        ).strip().lower()

        price = normalize_number(
            row.get("price")
        )

        # Explicitly unavailable
        if availability == "unavailable":

            unavailable_count += 1

            continue

        # Valid available product
        if (
            availability == "available"
            and price is not None
        ):

            available_rows.append(row)

    # --------------------------------------------------------
    # ALL VARIANTS UNAVAILABLE
    # --------------------------------------------------------

    if (
        not available_rows
        and unavailable_count > 0
    ):

        return "unavailable"

    # --------------------------------------------------------
    # No usable scraper result
    # --------------------------------------------------------

    if not available_rows:
        return ""

    # --------------------------------------------------------
    # Find all rows with a usable price
    # --------------------------------------------------------

    priced_rows = [
        row
        for row in available_rows
        if normalize_number(
            row.get("price")
        ) is not None
    ]

    if not priced_rows:
        return ""

    # --------------------------------------------------------
    # Find LOWEST available price
    # --------------------------------------------------------

    min_price = min(
        normalize_number(
            row["price"]
        )
        for row in priced_rows
    )

    # --------------------------------------------------------
    # Compare with RRP after Promotion
    # --------------------------------------------------------
    #
    # If the cheapest available product is exactly
    # the promotion price -> OK
    #
    # Example:
    #
    # Promotion = 1699
    # Website:
    #   Blue = 1699
    #   Black = unavailable
    #
    # Result:
    #   OK
    #
    # --------------------------------------------------------

    if promotion_price is not None:

        if min_price == promotion_price:

            return "OK"

    # --------------------------------------------------------
    # Find rows having the LOWEST price
    # --------------------------------------------------------

    min_rows = [
        row
        for row in priced_rows
        if normalize_number(
            row.get("price")
        ) == min_price
    ]

    # --------------------------------------------------------
    # Collect colours for the lowest price
    # --------------------------------------------------------

    colors = []

    for row in min_rows:

        color = str(
            row.get("variant", "")
        ).strip()

        if not color:
            color = "Unknown"

        # Avoid duplicate colour names
        if color.lower() not in {
            existing.lower()
            for existing in colors
        }:

            colors.append(color)

    # --------------------------------------------------------
    # Check whether ALL available variants have
    # exactly the same price
    # --------------------------------------------------------

    all_prices = [
        normalize_number(
            row["price"]
        )
        for row in priced_rows
    ]

    if len(set(all_prices)) == 1:

        if min_price.is_integer():

            return str(
                int(min_price)
            )

        return str(min_price)

    # --------------------------------------------------------
    # Different prices:
    # Return:
    #     <lowest price> <colour>
    # --------------------------------------------------------

    color_text = ", ".join(
        color.lower()
        for color in colors
    )

    if min_price.is_integer():

        price_text = str(
            int(min_price)
        )

    else:

        price_text = str(
            min_price
        )

    return f"{price_text} {color_text}"


# 10.3 READ WEBSITE HEADERS FROM RESULTS

website_columns = [
    results_sheet.cell(
        1,
        col
    ).value
    for col in range(
        8,
        16
    )
]

print(
    "Results website columns:"
)

print(
    website_columns
)


# 10.4 CREATE RAWDATA LOOKUP

raw_lookup = {}

for _, raw_row in results_df.iterrows():

    target_id = str(
        raw_row["target_id"]
    ).strip().upper()

    website = str(
        raw_row["website"]
    ).strip().upper()

    key = (
        target_id,
        website
    )

    raw_lookup.setdefault(
        key,
        []
    ).append(
        raw_row.to_dict()
    )


# 10.5 WRITE WEBSITE RESULTS

result_row_number = 2


for product_row in products_sheet.iter_rows(
    min_row=2,
    values_only=True
):

    if not product_row:
        continue

    # --------------------------------------------------------
    # CURRENT PRODUCTS STRUCTURE

    model = product_row[1]

    ram = product_row[3]

    storage = product_row[4]

    # --------------------------------------------------------
    # Skip invalid product rows
    # --------------------------------------------------------

    if (
        model is None
        or ram is None
        or storage is None
    ):

        continue

    model = str(
        model
    ).strip()

    ram = str(
        ram
    ).strip()

    storage = str(
        storage
    ).strip()

    # --------------------------------------------------------
    # Promotion price comes from Results

    promotion_price = results_sheet.cell(
        result_row_number,
        5
    ).value

    # --------------------------------------------------------
    # Construct TargetID
    # --------------------------------------------------------

    target_id = (
        f"{model}-{ram}-{storage}"
    ).upper()

    print()
    print(
        "=" * 60
    )

    print(
        f"Generating Results for: {target_id}"
    )

    print(
        f"Promotion price: {promotion_price}"
    )

    # --------------------------------------------------------
    # WRITE WEBSITE RESULTS H:O
    # --------------------------------------------------------

    for offset, website_column in enumerate(
        website_columns,
        start=8
    ):

        # Empty website header
        if not website_column:
            continue

        website_key = str(
            website_column
        ).strip().upper()

        # ----------------------------------------------------
        # Find scraper results
        # ----------------------------------------------------

        raw_rows = raw_lookup.get(
            (
                target_id,
                website_key
            ),
            []
        )

        # ----------------------------------------------------
        # Build final report value
        # ----------------------------------------------------

        report_value = build_report_value(
            raw_rows,
            promotion_price
        )

        # ----------------------------------------------------
        # WRITE ONLY H:O
        # ----------------------------------------------------

        results_sheet.cell(
            result_row_number,
            offset
        ).value = report_value

        print(
            f"  {website_column}: {report_value}"
        )

    # --------------------------------------------------------
    # Next product row
    # --------------------------------------------------------

    result_row_number += 1


# 10.6 FINISHED

print()
print(
    "=" * 70
)

print(
    "WEBSITE RESULTS GENERATED"
)

print(
    "=" * 70
)

print(
    "Rows processed:",
    result_row_number - 2
)

print(
    "Website columns written:",
    website_columns
)

print(
    "Columns A:G were NOT modified."
)

print(
    "Only columns H:O were written."
)


# 10.7 PREVIEW

report_preview = pd.DataFrame(
    results_sheet.values
)

display(
    report_preview
)

Results website columns:
['MEX', 'Electro.pl', 'Avans', 'MSH', 'KTR', 'XKOM', 'NEONET', 'Max electro']

Generating Results for: SOMALIAA-4-64
Promotion price: 399
  MEX: OK
  Electro.pl: OK
  Avans: OK
  MSH: 399.99 black
  KTR: OK
  XKOM: 449
  NEONET: 449
  Max electro: OK

Generating Results for: P15AE-4-128
Promotion price: 499
  MEX: OK
  Electro.pl: unavailable
  Avans: OK
  MSH: 499.99 black
  KTR: OK
  XKOM: unavailable
  NEONET: unavailable
  Max electro: unavailable

Generating Results for: O19AE-6-128
Promotion price: 649
  MEX: OK
  Electro.pl: unavailable
  Avans: unavailable
  MSH: 629 grey
  KTR: OK
  XKOM: unavailable
  NEONET: unavailable
  Max electro: unavailable

Generating Results for: O19AE-8-256
Promotion price: 749
  MEX: OK
  Electro.pl: OK
  Avans: OK
  MSH: 719
  KTR: unavailable
  XKOM: unavailable
  NEONET: unavailable
  Max electro: unavailable

Generating Results for: P16U-12-512
Promotion price: 1899
  MEX: OK
  Electro.pl: 1795 black
  Avans: 1795.39 bl

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,SKU,Memory,Product Name,RRP,RRP after Promotion,Offer start date,Offer end date,MEX,Electro.pl,Avans,MSH,KTR,XKOM,NEONET,Max electro
1,SOMALIAA,4+64,Redmi A7 pro,549,399,2026-08-17 00:00:00,2026-08-23 00:00:00,OK,OK,OK,399.99 black,OK,449,449,OK
2,P15AE,4+128,Redmi 15C,499,499,2026-08-17 00:00:00,2026-08-23 00:00:00,OK,unavailable,OK,499.99 black,OK,unavailable,unavailable,unavailable
3,O19AE,6+128,Redmi 15,649,649,2026-08-17 00:00:00,2026-08-23 00:00:00,OK,unavailable,unavailable,629 grey,OK,unavailable,unavailable,unavailable
4,O19AE,8+256,Redmi 15,749,749,2026-08-17 00:00:00,2026-08-23 00:00:00,OK,OK,OK,719,unavailable,unavailable,unavailable,unavailable
5,P16U,12+512,Redmi Note 15 Pro+ 5G,2299,1899,2026-08-17 00:00:00,2026-08-23 00:00:00,OK,1795 black,1795.39 black,unavailable,OK,unavailable,,unavailable
6,P16U,8+256,Redmi Note 15 Pro+ 5G,1999,1699,2026-08-17 00:00:00,2026-08-23 00:00:00,OK,1531 blue,1531.43 blue,OK,OK,OK,OK,OK
7,P16,8+256,Redmi Note 15 Pro 5G,1699,1399,2026-08-17 00:00:00,2026-08-23 00:00:00,1398 titanium,1398 titanium,1398 titanium,OK,1398 unknown,1499,1398,OK
8,P17,6+128,Redmi Note 15 5G,1199,1199,2026-08-17 00:00:00,2026-08-23 00:00:00,976,982,"982.64 black, purple",979,1029,OK,OK,899
9,P6,8+256,Redmi Note 15 Pro,1499,1299,2026-08-17 00:00:00,2026-08-23 00:00:00,OK,OK,OK,OK,OK,"1399 black, blue",1398 blue,OK


In [26]:
# ============================================================
# 11. SAVE WORKBOOK
# ============================================================

workbook.save(excel_file)

print("RawData and Results updated successfully.")
print("Workbook saved:", excel_file)


RawData and Results updated successfully.
Workbook saved: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\XM_Price_Checker_Python.xlsm
